# Fine-tune YAMNet — D2

**Pendekatan A.** Membuka bobot YAMNet dan melatihnya di data sirine.

YAMNet TF-Hub tidak bisa di-fine-tune langsung (grafik inference, input tak ter-batch).
Solusi: **rebuild core YAMNet sebagai Keras** (arsitektur & nama layer sama persis dengan
implementasi resmi), **muat bobot pretrained** `yamnet.h5` via `load_weights(by_name=True)`,
lalu:

```
audio 16k → log-mel patches (6×96×64) → TimeDistributed(core YAMNet) → (6×1024)
          → pool mean+std → head MLP → 4 kelas
```

Two-fase: (1) core beku, latih head; (2) buka core, LR kecil.

- Jalan di **Kaggle GPU** (full) & CPU lokal (`SMOKE=True`, uji bebas-bug).
- Test set identik dgn lokal (240). Pembanding: frozen meanstd **0.7983**.

In [1]:
import os
os.environ["TF_USE_LEGACY_KERAS"] = "1"   # yamnet.h5 = Keras 2; tf.keras -> Keras 2
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")

import json, time
from types import SimpleNamespace
from pathlib import Path
import numpy as np
import pandas as pd
import librosa
import tensorflow as tf
from sklearn.metrics import f1_score, accuracy_score, classification_report, confusion_matrix

SEED = 42
tf.keras.utils.set_random_seed(SEED)
layers = tf.keras.layers

KAGGLE = Path("/kaggle/input").exists()
SMOKE = not KAGGLE
CLASSES = ["ambulance", "firetruck", "police", "traffic"]
YAMNET_SR = 16000
N_SAMPLES = int(YAMNET_SR * 3.0)          # 48000 -> 5 patch penuh (dinamis)

# hyperparameter YAMNet resmi (params.py)
P = SimpleNamespace(
    sample_rate=16000.0, stft_window_seconds=0.025, stft_hop_seconds=0.010,
    mel_bands=64, mel_min_hz=125.0, mel_max_hz=7500.0, log_offset=0.001,
    patch_window_seconds=0.96, patch_hop_seconds=0.48,
    conv_padding="same", batchnorm_center=True, batchnorm_scale=False,
    batchnorm_epsilon=1e-4)
P.patch_frames = int(round(P.patch_window_seconds / P.stft_hop_seconds))  # 96
P.patch_bands = P.mel_bands

if KAGGLE:
    inp = Path("/kaggle/input")
    cand = [p for p in inp.rglob("*") if p.is_dir() and p.name == "ambulance"]
    DATA_ROOT = cand[0].parent if cand else inp
    SPLIT_DIR = next(inp.rglob("split_d2_train.csv")).parent
    OUT = Path("/kaggle/working")
else:
    ROOT = Path.cwd().parent.parent if Path.cwd().name == "kaggle" else \
           (Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd())
    DATA_ROOT = ROOT / "Dataset" / "Dataset2"
    SPLIT_DIR = ROOT / "ml"
    OUT = ROOT / "ml" / "artifacts" / "d2_yamnet_finetune"
OUT.mkdir(parents=True, exist_ok=True)

YAMNET_H5 = tf.keras.utils.get_file(
    "yamnet.h5", "https://storage.googleapis.com/audioset/yamnet.h5")
print(f"KAGGLE={KAGGLE} · SMOKE={SMOKE} · GPU={tf.config.list_physical_devices('GPU')}")
print(f"weights: {YAMNET_H5}")

KAGGLE=False · SMOKE=True · GPU=[]
weights: C:\Users\ABIL KHOIRI\.keras\datasets\yamnet.h5


## 1 · Resolusi path + join ke split (identik dgn notebook CNN)

In [2]:
wav_index = {}
for p in DATA_ROOT.rglob("*.wav"):
    if p.parent.name in CLASSES:
        wav_index[(p.parent.name, p.name)] = str(p)


def load_split(name):
    df = pd.read_csv(SPLIT_DIR / f"split_d2_{name}.csv")
    df["path"] = [wav_index.get((l, f)) for l, f in zip(df.label, df.filename)]
    assert df.path.isna().sum() == 0, f"file split '{name}' tak ketemu"
    return df


tr, va, te = load_split("train"), load_split("val"), load_split("test")
assert (len(tr), len(va), len(te)) == (1195, 240, 240) or SMOKE
if SMOKE:
    tr = tr.groupby("label").head(12).reset_index(drop=True)
    va = va.groupby("label").head(8).reset_index(drop=True)
    te = te.groupby("label").head(8).reset_index(drop=True)
print(f"train {len(tr)} · val {len(va)} · test {len(te)}")

train 48 · val 32 · test 32


## 2 · Fitur = log-mel patches YAMNet (vendored dari features.py resmi)

Fungsi ini identik dengan implementasi resmi YAMNet supaya patch cocok dengan bobot
pretrained. Tiap klip 3 s → **(6, 96, 64)**.

In [3]:
def _wave_to_patches(waveform):
    """Vendored dari yamnet/features.py — waveform 16k -> patches (n,96,64)."""
    win = int(round(P.sample_rate * P.stft_window_seconds))
    hop = int(round(P.sample_rate * P.stft_hop_seconds))
    fft_len = 2 ** int(np.ceil(np.log(win) / np.log(2.0)))
    nbins = fft_len // 2 + 1
    mag = tf.abs(tf.signal.stft(waveform, frame_length=win, frame_step=hop,
                                fft_length=fft_len))
    mel_w = tf.signal.linear_to_mel_weight_matrix(
        num_mel_bins=P.mel_bands, num_spectrogram_bins=nbins,
        sample_rate=P.sample_rate, lower_edge_hertz=P.mel_min_hz,
        upper_edge_hertz=P.mel_max_hz)
    mel = tf.matmul(mag, mel_w)
    log_mel = tf.math.log(mel + P.log_offset)
    spec_rate = P.sample_rate / hop
    pw = int(round(spec_rate * P.patch_window_seconds))
    ph = int(round(spec_rate * P.patch_hop_seconds))
    feats = tf.signal.frame(log_mel, frame_length=pw, frame_step=ph, axis=0)
    return feats                                   # (n_patches, 96, 64)


def load_raw(path):
    """Audio 16k mono, panjang tetap N_SAMPLES (belum diaugmentasi)."""
    y, _ = librosa.load(path, sr=YAMNET_SR, mono=True)
    if len(y) < N_SAMPLES:
        y = np.pad(y, (0, N_SAMPLES - len(y)))
    return y[:N_SAMPLES].astype(np.float32)


def patches_from_wave(y):
    return _wave_to_patches(tf.constant(y)).numpy()


def load_patches(path):
    return patches_from_wave(load_raw(path))


N_PATCH = load_patches(tr.iloc[0].path).shape[0]
print(f"patch per klip: {N_PATCH} (dinamis — dipakai apa adanya, sama untuk semua klip 3 s)")


idx = {c: i for i, c in enumerate(CLASSES)}

def build_xy(df):
    X = np.stack([load_patches(p) for p in df.path]).astype(np.float32)  # (N,n_patch,96,64)
    return X[..., None], df.label.map(idx).to_numpy()                    # +channel


t0 = time.time()
Xtr, ytr = build_xy(tr); Xva, yva = build_xy(va); Xte, yte = build_xy(te)
print(f"fitur: train {Xtr.shape} · val {Xva.shape[0]} · test {Xte.shape[0]} "
      f"({time.time()-t0:.0f}s)")

patch per klip: 5 (dinamis — dipakai apa adanya, sama untuk semua klip 3 s)


fitur: train (48, 5, 96, 64, 1) · val 32 · test 32 (2s)


## 3 · Rebuild core YAMNet + muat bobot pretrained

Layer & namanya sama persis dgn `yamnet/yamnet.py` resmi, jadi `load_weights(by_name=True)`
mengisi bobot conv/bn pretrained. Head klasifikasi 521-kelas asli **tidak** dipakai.

In [4]:
def _bn(name):
    return layers.BatchNormalization(name=name, center=P.batchnorm_center,
                                     scale=P.batchnorm_scale, epsilon=P.batchnorm_epsilon)

def _conv(name, kernel, stride, filters):
    def f(x):
        x = layers.Conv2D(name=f"{name}/conv", filters=filters, kernel_size=kernel,
                          strides=stride, padding=P.conv_padding, use_bias=False)(x)
        x = _bn(f"{name}/conv/bn")(x)
        return layers.ReLU(name=f"{name}/relu")(x)
    return f

def _sep(name, kernel, stride, filters):
    def f(x):
        x = layers.DepthwiseConv2D(name=f"{name}/depthwise_conv", kernel_size=kernel,
                                   strides=stride, depth_multiplier=1,
                                   padding=P.conv_padding, use_bias=False)(x)
        x = _bn(f"{name}/depthwise_conv/bn")(x)
        x = layers.ReLU(name=f"{name}/depthwise_conv/relu")(x)
        x = layers.Conv2D(name=f"{name}/pointwise_conv", filters=filters, kernel_size=(1, 1),
                          strides=1, padding=P.conv_padding, use_bias=False)(x)
        x = _bn(f"{name}/pointwise_conv/bn")(x)
        return layers.ReLU(name=f"{name}/pointwise_conv/relu")(x)
    return f

_DEFS = [(_conv,[3,3],2,32),(_sep,[3,3],1,64),(_sep,[3,3],2,128),(_sep,[3,3],1,128),
         (_sep,[3,3],2,256),(_sep,[3,3],1,256),(_sep,[3,3],2,512),(_sep,[3,3],1,512),
         (_sep,[3,3],1,512),(_sep,[3,3],1,512),(_sep,[3,3],1,512),(_sep,[3,3],1,512),
         (_sep,[3,3],2,1024),(_sep,[3,3],1,1024)]


def build_core():
    inp = layers.Input(shape=(P.patch_frames, P.patch_bands, 1))
    net = inp
    for i, (fun, k, s, f) in enumerate(_DEFS):
        net = fun(f"layer{i+1}", k, s, f)(net)
    emb = layers.GlobalAveragePooling2D()(net)          # (1024,)
    return tf.keras.Model(inp, emb, name="yamnet_core")


core = build_core()
w_before = float(core.get_layer("layer1/conv").weights[0].numpy().std())
core.load_weights(YAMNET_H5, by_name=True, skip_mismatch=True)
w_after = float(core.get_layer("layer1/conv").weights[0].numpy().std())
print(f"core params: {core.count_params():,}")
print(f"bobot layer1 std sebelum={w_before:.4f} sesudah={w_after:.4f} "
      f"(berubah => pretrained termuat: {abs(w_after-w_before)>1e-6})")
assert abs(w_after - w_before) > 1e-6, "bobot pretrained TIDAK termuat!"

core params: 3,217,344
bobot layer1 std sebelum=0.0813 sesudah=0.0181 (berubah => pretrained termuat: True)


## 4 · Model lengkap — TimeDistributed core + pool mean+std + head

In [5]:
@tf.keras.utils.register_keras_serializable()
class MeanStdPool(layers.Layer):
    """Pool antar-patch: concat(mean, std). Layer proper (bukan Lambda) supaya model
    bisa di-load_model TANPA safe_mode — penting untuk deployment."""
    def call(self, x):                                  # (B, n_patch, 1024)
        mean = tf.reduce_mean(x, axis=1)
        var = tf.reduce_mean(tf.square(x - mean[:, None, :]), axis=1)
        std = tf.sqrt(var + 1e-6)                        # eps: cegah NaN gradient saat std=0
        return tf.concat([mean, std], axis=1)           # (B, 2048)


def build_model():
    inp = layers.Input(shape=(N_PATCH, P.patch_frames, P.patch_bands, 1))
    x = layers.TimeDistributed(core)(inp)               # (B, n_patch, 1024)
    x = MeanStdPool()(x)                                 # (B, 2048)
    x = layers.Dense(256, activation="relu",
                     kernel_regularizer=tf.keras.regularizers.l2(1e-4))(x)   # A: L2
    x = layers.Dropout(0.5)(x)                           # A: dropout 0.3 -> 0.5
    out = layers.Dense(len(CLASSES), activation="softmax")(x)
    return tf.keras.Model(inp, out)


def set_core_finetune(on):
    """on=False: seluruh core beku. on=True: HANYA blok teratas (layer12-14) yang
    dilatih; BatchNorm tetap beku. Buka semua backbone di data kecil = overfit parah."""
    if not on:
        core.trainable = False
        return
    core.trainable = True
    TOP = ("layer10", "layer11", "layer12", "layer13", "layer14")   # D: buka lebih banyak
    for lyr in core.layers:
        if isinstance(lyr, layers.BatchNormalization):
            lyr.trainable = False                        # BN beku saat fine-tune
        elif not lyr.name.startswith(TOP):
            lyr.trainable = False                        # hanya blok teratas terbuka


class ValMacroF1(tf.keras.callbacks.Callback):
    def __init__(self, Xv, yv): super().__init__(); self.Xv, self.yv = Xv, yv
    def on_epoch_end(self, epoch, logs=None):
        logs = logs if logs is not None else {}
        p = self.model.predict(self.Xv, verbose=0).argmax(1)
        logs["val_macro_f1"] = f1_score(self.yv, p, average="macro")


def cbs():
    return [ValMacroF1(Xva, yva),
            tf.keras.callbacks.EarlyStopping(monitor="val_macro_f1", mode="max",
                                             patience=6 if SMOKE else 12,
                                             restore_best_weights=True)]

## 5 · Two-phase training

In [6]:
E1, E2 = (2, 2) if SMOKE else (12, 40)
BS = 16
P1 = str(OUT / "phase1_best.weights.h5")

# --- Fase 1: core beku, latih head. Model dibangun SEKALI (head dipertahankan). ---
set_core_finetune(False)
model = build_model()
model.compile(optimizer=tf.keras.optimizers.Adam(1e-3, clipnorm=1.0),
              loss="sparse_categorical_crossentropy", metrics=["accuracy"])
print("=== Fase 1: head warm-up (core beku) ===")
h1 = model.fit(Xtr, ytr, validation_data=(Xva, yva), epochs=E1, batch_size=BS,
               callbacks=cbs(), verbose=2)              # restore_best_weights -> model=best fase1
best1 = max(h1.history["val_macro_f1"])
model.save_weights(P1)                                  # simpan bobot terbaik fase 1

# --- Fase 2: buka HANYA blok teratas, LR sangat kecil (1e-5), BN beku ---
set_core_finetune(True)
model.compile(optimizer=tf.keras.optimizers.Adam(1e-5, clipnorm=1.0),
              loss="sparse_categorical_crossentropy", metrics=["accuracy"])
print(f"\ntrainable params fase 2: {int(np.sum([np.prod(w.shape) for w in model.trainable_weights])):,}")
print("=== Fase 2: fine-tune blok teratas (LR 1e-5, BN beku) ===")
h2 = model.fit(Xtr, ytr, validation_data=(Xva, yva), epochs=E2, batch_size=BS,
               callbacks=cbs(), verbose=2)
best2 = max(h2.history["val_macro_f1"])

# --- ambil yang terbaik LINTAS kedua fase: tak mungkin lebih buruk dari fase 1 ---
if best1 >= best2:
    model.load_weights(P1)
    print(f"\n-> pakai bobot FASE 1 (val_macro_f1 {best1:.4f} >= fase2 {best2:.4f})")
else:
    print(f"\n-> pakai bobot FASE 2 (val_macro_f1 {best2:.4f} > fase1 {best1:.4f}) — fine-tune membantu!")

=== Fase 1: head warm-up (core beku) ===
Epoch 1/2


3/3 - 30s - loss: 1.4183 - accuracy: 0.4375 - val_loss: 1.1612 - val_accuracy: 0.5625 - val_macro_f1: 0.4762 - 30s/epoch - 10s/step


Epoch 2/2


3/3 - 0s - loss: 1.1316 - accuracy: 0.5417 - val_loss: 1.0246 - val_accuracy: 0.5312 - val_macro_f1: 0.5263 - 369ms/epoch - 123ms/step



trainable params fase 2: 2,912,516
=== Fase 2: fine-tune blok teratas (LR 1e-5, BN beku) ===
Epoch 1/2


3/3 - 19s - loss: 0.7809 - accuracy: 0.8125 - val_loss: 0.9683 - val_accuracy: 0.5312 - val_macro_f1: 0.5299 - 19s/epoch - 6s/step


Epoch 2/2


3/3 - 0s - loss: 0.8433 - accuracy: 0.7708 - val_loss: 0.9290 - val_accuracy: 0.5938 - val_macro_f1: 0.5977 - 484ms/epoch - 161ms/step



-> pakai bobot FASE 2 (val_macro_f1 0.5977 > fase1 0.5263) — fine-tune membantu!


## 6 · Evaluasi + simpan

In [7]:
y_pred = model.predict(Xte, verbose=0).argmax(1)
macro_f1 = f1_score(yte, y_pred, average="macro")
acc = accuracy_score(yte, y_pred)
print(f"TEST macro-F1 : {macro_f1:.4f}   (frozen meanstd 0.7983 · target 0.85)")
print(f"TEST accuracy : {acc:.4f}\n")
print(classification_report(yte, y_pred, target_names=CLASSES, digits=3))
print("confusion:\n", confusion_matrix(yte, y_pred))

model.save(OUT / "model.keras")
# verifikasi bisa di-load ulang TANPA safe_mode (syarat deploy bersih)
_r = tf.keras.models.load_model(OUT / "model.keras")
print("reload OK tanpa safe_mode — siap deploy")
json.dump({
    "exp_id": "d2_yamnet_finetune", "backbone": "yamnet_finetuned",
    "feature": "yamnet_patches_meanstd", "test_macro_f1": float(macro_f1),
    "test_accuracy": float(acc), "smoke": SMOKE,
    "per_class_f1": dict(zip(CLASSES, f1_score(yte, y_pred, average=None).tolist())),
}, open(OUT / "metrics.json", "w"), indent=2)
print(f"\ntersimpan -> {OUT}")
if SMOKE:
    print("\n[SMOKE] angka tidak bermakna — hanya uji bebas-bug. Jalankan penuh di Kaggle GPU.")

TEST macro-F1 : 0.5438   (frozen meanstd 0.7983 · target 0.85)
TEST accuracy : 0.5625

              precision    recall  f1-score   support

   ambulance      0.429     0.375     0.400         8
   firetruck      0.455     0.625     0.526         8
      police      0.400     0.250     0.308         8
     traffic      0.889     1.000     0.941         8

    accuracy                          0.562        32
   macro avg      0.543     0.562     0.544        32
weighted avg      0.543     0.562     0.544        32

confusion:
 [[3 2 3 0]
 [3 5 0 0]
 [1 4 2 1]
 [0 0 0 8]]


reload OK tanpa safe_mode — siap deploy

tersimpan -> D:\Coding Vscode\Siren Classification\ml\artifacts\d2_yamnet_finetune

[SMOKE] angka tidak bermakna — hanya uji bebas-bug. Jalankan penuh di Kaggle GPU.


---
Bandingkan `test_macro_f1` dengan notebook CNN fine-tune & frozen meanstd (0.7983). Pemenang
diekspor end-to-end untuk deployment (Fase 4).